# Zep Memory Application

1) Initialize Client  

2) Create User in Zep  

3) Create Thread ID                 ---- Thread is one Conversation Session  

4) Add messages to zep Memory along with the context block                 ------ context_block = LLM-ready memory summary  

5) Can retrieve the context Memory  

6) Can save events. can retrive those events                               ------ event is structured memory  

In [ ]:
## 1) Install the Zep SDK

# pip install zep-cloud

In [17]:
## 2) Initialize the Zep client

from zep_cloud.client import Zep
from openai import OpenAI

client = Zep(api_key = 'API_KEY')
openai_client = OpenAI(api_key = 'API_KEY')

print('Initialized!')


Initialized!


In [18]:
## 3) Create a Zep user for each of your users

user_id = "Jai"

client.user.add(
    user_id=user_id,
    first_name="Jai",
    email="jai@gmail.com"
)

print('User Created!')


User Created!


In [19]:
## 4) Create a Zep thread for each of your threads

import uuid

thread_id = uuid.uuid4().hex                                # A new thread identifier

client.thread.create(
    thread_id=thread_id,
    user_id=user_id,
)

print('Thread Created!')

Thread Created!


In [20]:
## 5) Add incoming user messages to Zep

from zep_cloud.types import Message
from datetime import datetime, timezone

messages = [
    Message(
        created_at=datetime.now(timezone.utc).isoformat(),
        name="Jai",
        role="user",
        content="""Hi, I’m 27 years old.
                    I was diagnosed with diabetes 2 years ago.
                    I started taking Metformin 6 months back.
                    Last week I had severe headaches for 3 days.
                    Yesterday I visited Dr. Ravi at Apollo Hospital.
                    He advised me to do a blood sugar test tomorrow morning.
                    My next follow-up appointment is scheduled for next Monday at 10 AM.
                    Also, I’m allergic to penicillin.""",
    )
]

response = client.thread.add_messages(thread_id, messages=messages)

print("Conversation Stored!")

Conversation Stored!


In [31]:
## 6) Retrieve Zep context block

# Get context for the thread
user_context = client.thread.get_user_context(thread_id=thread_id)

# Access the context block
context_block = user_context.context
print(context_block)




# This is the user summary
<USER_SUMMARY>
The user is 27 years old and was diagnosed with diabetes 2 years ago. They have been taking Metformin for 6 months. The user is allergic to penicillin. They experienced severe headaches for 3 days last week.
</USER_SUMMARY>


FACTS and ENTITIES represent relevant context to the current conversation.

# These are the most relevant facts and their valid date ranges
# format: FACT (Date range: from - to)
# NOTE: Facts ending in "present" are currently valid (e.g., "Jane prefers her coffee with milk (2024-01-15 10:30:00 - present)" means Jane currently prefers coffee with milk)
#       Facts with a past end date used to be valid but are NOT CURRENTLY VALID (e.g., "Jane prefers her coffee with milk (2024-01-15 10:30:00 - 2024-06-20 14:00:00)" means Jane no longer prefers coffee with milk)
<FACTS>
  - Jai has a follow-up appointment next Monday at 10 AM. (2026-04-29 10:00:00 - present)
  - Jai is allergic to penicillin. (2026-04-27 04:05:56 - prese

In [ ]:
def chat_with_agent(user_input):

    # ---- Store user message in Zep ----
    user_msg = Message(
        created_at=datetime.now(timezone.utc).isoformat(),
        name="Jai",
        role="user",
        content=user_input,
    )

    client.thread.add_messages(thread_id, messages=[user_msg])

    # ---- Get context from Zep ----
    user_context = client.thread.get_user_context(thread_id=thread_id)
    context_block = user_context.context

    # ---- Send to OpenAI ----
    response = openai_client.chat.completions.create(
        model="gpt-4.1-mini",   
        messages=[
            {
                "role": "system",
                "content": context_block
            },
            {
                "role": "user",
                "content": user_input
            }
        ]
    )

    answer = response.choices[0].message.content

    # ---- Store assistant response in Zep ----
    assistant_msg = Message(
        created_at=datetime.now(timezone.utc).isoformat(),
        name="AI Assistant",
        role="assistant",
        content=answer,
    )

    client.thread.add_messages(thread_id, messages=[assistant_msg])

    return answer

In [33]:
print("\n🤖 AI Agent Ready! Type 'exit' to stop.\n")


n = 1

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    reply = chat_with_agent(user_input)
    print(n, user_input)
    print("AI:", reply)
    print('---------------------------------------')
    n += 1


🤖 AI Agent Ready! Type 'exit' to stop.

1 What is my name/
AI: Your name is Jai.
---------------------------------------
2 How old am I?
AI: You are 27 years old.
---------------------------------------
3 Do I have any health issues?
AI: Based on the information you’ve shared, you have been diagnosed with diabetes for the past 2 years and have been taking Metformin for 6 months to manage it. Additionally, you experienced severe headaches for 3 days last week. You are also allergic to penicillin, which is important to keep in mind for any medication.

If you have any concerns about your symptoms or health, it’s a good idea to discuss them with your healthcare provider. You also have a follow-up appointment next Monday at 10 AM, which would be a good opportunity to review your health status.
---------------------------------------
4 do I have to visit a doctor?
AI: Since you had severe headaches for 3 days last week and you have a follow-up appointment scheduled for next Monday at 10 AM

In [35]:
messages = [
    Message(
        created_at=datetime.now(timezone.utc).isoformat(),
        name="Jai",
        role="user",
        content="""I was living in Hyderabad and working remotely as a software engineer. 
        I used to wake up at 9 AM, skip breakfast, and work late nights until 2 AM.
        I moved from Hyderabad to Bangalore for a new job in March 2025. 
        My office required me to work onsite, so I started waking up at 6 AM and began eating breakfast regularly.
        I relocated again to Pune in January 2026, because my company opened a new branch there. 
        I started going to the gym in the evenings and reduced my coffee intake from 5 cups a day to 2 cups.
""",
    )
]
response = client.thread.add_messages(thread_id, messages=messages)


print("Conversation Stored!")

Conversation Stored!


In [36]:
print("\n🤖 AI Agent Ready! Type 'exit' to stop.\n")


n = 1

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    reply = chat_with_agent(user_input)
    print(n, user_input)
    print("AI:", reply)
    print('---------------------------------------')
    n += 1


🤖 AI Agent Ready! Type 'exit' to stop.

1 Where do I currently live?
AI: You currently live in Pune.
---------------------------------------
2 did I change my location recently
AI: Yes, you recently changed your location. You relocated to Pune in January 2026 after previously moving to Bangalore in March 2025.
---------------------------------------
3 do you know about my routine?
AI: Yes, I do know about your routine! You wake up at 6 AM, eat breakfast regularly, limit your coffee intake to 2 cups per day, and you exercise at the gym in the evenings. If you'd like, I can help you keep track of or adjust your routine. Would you like any specific reminders or suggestions?
---------------------------------------


In [44]:
## Add streaming business data to Zep

import json

# Example: User listened to a song in your application
event_data = {
    "user_id": "Jai",
    "user_name": "Jai",
    "event_type": "song_played",
    "song_title": "Bohemian Rhapsody",
    "artist": "Queen",
    "duration_seconds": 354
}

results = client.graph.add(
    user_id="Jai",
    type="json",
    data=json.dumps(event_data)
)

print('Added!')

Added!


In [45]:
print(results)

content='{"user_id":"Jai","user_name":"Jai","event_type":"song_played","song_title":"Bohemian Rhapsody","artist":"Queen","duration_seconds":354}' created_at='2026-04-27T04:14:31.940959Z' metadata=None processed=False relevance=None role=None role_type=None score=None selection_rank=None source='json' source_description='' task_id=None thread_id=None uuid_='cb5d78ac-26bd-4d63-99e0-bc07dda0ad35' session_id=None


In [ ]:
## Deleting the user id

client.user.delete(user_id=user_id)
print("✅ User and all memory deleted")

✅ User and all memory deleted


# Custom Context

In [ ]:
import uuid
from zep_cloud.client import Zep
from zep_cloud.types import Message
from openai import OpenAI

print('Initialized!')

user_id = "John"

zep_client.user.add(
    user_id=user_id,
    first_name="John",
    email="john@gmail.com"
)

thread_id = uuid.uuid4().hex                                

zep_client.thread.create(
    thread_id=thread_id,
    user_id=user_id,
)

print('Thread Created!')

messages = [

    # Personal info
    Message(
        role="user",
        content="My name is John Miller"
    ),
    Message(
        role="user",
        content="I am 45 years old"
    ),
    Message(
        role="user",
        content="I live in Hyderabad, India"
    ),
    Message(
        role="user",
        content="I work as a software engineer"
    ),
    Message(
        role="user",
        content="I am married and have two children"
    ),

    # Medical history
    Message(
        role="user",
        content="I was diagnosed with diabetes last year"
    ),
    Message(
        role="user",
        content="I also have high blood pressure"
    ),
    Message(
        role="user",
        content="I take Metformin daily"
    ),
    Message(
        role="user",
        content="I take Amlodipine for blood pressure"
    ),
    Message(
        role="user",
        content="I am allergic to penicillin"
    ),
    Message(
        role="user",
        content="I had knee surgery in 2022"
    ),

    # Appointments
    Message(
        role="user",
        content="I visited Dr. Smith last week"
    ),
    Message(
        role="user",
        content="My next appointment is on May 5th"
    ),

    # Lifestyle
    Message(
        role="user",
        content="I go for morning walks every day"
    ),
    Message(
        role="user",
        content="I avoid eating sugar"
    ),
    Message(
        role="user",
        content="I love biryani and pizza"
    ),

    # Random preferences
    Message(
        role="user",
        content="My favorite travel destination is Goa"
    ),
    Message(
        role="user",
        content="I prefer morning appointments over evening appointments"
    )
]


zep_client.thread.add_messages(
    thread_id=thread_id,
    messages=messages
)

print("Conversation stored successfully")


Initialized!
Thread Created!
Conversation stored successfully


In [79]:
print(messages)

[Message(content='\nMy name is John Miller.\nI am 45 years old.\nI live in Hyderabad, India.\nI work as a software engineer.\nI am married and have two children.\n\nI was diagnosed with diabetes last year.\nI also have high blood pressure.\nI take Metformin daily.\nI take Amlodipine for blood pressure.\nI am allergic to penicillin.\nI had knee surgery in 2022.\n\nI visited Dr. Smith last week.\nMy next appointment is on May 5th.\n\nI go for morning walks every day.\nI avoid eating sugar.\nI love biryani and pizza.\n\nMy favorite travel destination is Goa.\nI prefer morning appointments over evening appointments.\n\nI usually sleep at 11 PM.\nI drink coffee every morning.\nI recently started learning yoga.\nMy wife is a teacher.\nMy children study in primary school.\nI own a pet dog named Bruno.\nI frequently travel to Bangalore for work meetings.\n', created_at=None, metadata=None, name=None, processed=None, role='user', uuid_=None)]


In [97]:
search_results = zep_client.graph.search(
    user_id=user_id,
    query="medical history"
)

relevant_facts = []

for edge in search_results.edges:
    relevant_facts.append(edge.fact)

print(relevant_facts)

['John takes Amlodipine for a medical condition.', 'John visited Dr. Smith last week.', 'John has high blood pressure.', 'John is allergic to penicillin.', 'John has an appointment scheduled.', 'John takes Metformin daily.', 'John was diagnosed with diabetes last year.', 'John takes Amlodipine for blood pressure.', 'John is married.', 'The appointment is scheduled for May 5th.']


In [98]:
retrieved_memories = []

for result in relevant_facts:
    retrieved_memories.append(result)

custom_context = f"""
PATIENT HEALTH CONTEXT:
{retrieved_memories}
"""

print(custom_context)


PATIENT HEALTH CONTEXT:
['John takes Amlodipine for a medical condition.', 'John visited Dr. Smith last week.', 'John has high blood pressure.', 'John is allergic to penicillin.', 'John has an appointment scheduled.', 'John takes Metformin daily.', 'John was diagnosed with diabetes last year.', 'John takes Amlodipine for blood pressure.', 'John is married.', 'The appointment is scheduled for May 5th.']



In [99]:
user_question = "Can I eat sweets regularly?"

prompt = f"""

{custom_context}

Answer this question:
{user_question}
"""

response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response.choices[0].message.content)

As John has diabetes, it is generally advisable to limit the intake of sweets and sugary foods to maintain stable blood sugar levels. It's important for John to monitor his carbohydrate intake and make healthy food choices. Consulting with a healthcare provider or a dietitian can provide personalized advice on how to incorporate sweets in moderation while managing diabetes effectively.
